# 04 · Six-label redesign and final model (v4)

The v1 results showed that many of the 16 emotions were either too rare or too easily confused to label reliably.
This notebook rebuilds the pipeline around a smaller, more learnable label set.

* **Merge 16 emotions into 6:** PositiveAffect (Joy, Love, Gratitude), Contentment,
  NegativeAffect (Anger, Frustration), Disappointment, Fear, Disgust.
  Seven emotions (Hope, Interest, Surprise, Guilt, Pride, Sadness, Relief) are dropped.
* New prompt (example cues rather than an exhaustive checklist), targeted collection of reviews for under-represented
  classes, and a rebuilt holdout (1,504 reviews) that is labeled by hand outside this notebook.
* **Gate check:** the notebook refuses to continue until the human-labeled holdout exists.
* Re-label with GPT-4.1-mini (3 runs + majority vote): macro F1 **0.892** against the human labels.
* Re-label 17,145 train and 4,287 test reviews, then fine-tune RoBERTa (4 configs, weighted BCE or focal loss,
  max length 160) over **10 classes** (6 emotions + 4 evaluation dimensions).

**Result:** best macro F1 **0.878** on the holdout, against 0.600 for v1. Part of that gain comes from the easier
label set, so read the comparison with that in mind (see the repository README).

**Needs:** a GPU, `OPENAI_API_KEY`, and `final_holdout_v2.csv` (human labels, not included).


In [ ]:
# ── Setup (runs locally or on Colab) ─────────────────────────────────────────
# Install dependencies first:  pip install -r requirements.txt
import os

# Where datasets, LLM labels, checkpoints and models are written (not committed to git).
# Colab: mount Drive and set CAPSTONE_DATA_DIR to a Drive folder to persist between sessions.
DATA_DIR = os.environ.get("CAPSTONE_DATA_DIR", "data")
os.makedirs(DATA_DIR, exist_ok=True)
OUTPUT_DIR = DATA_DIR

# API keys: environment variables first, Colab Secrets as a fallback.
_ENV_NAMES = {
    "claude": "ANTHROPIC_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "xai": "XAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "minimax": "MINIMAX_API_KEY",
}

def get_api_key(name):
    value = os.environ.get(_ENV_NAMES.get(name, name)) or os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata as _colab_userdata
        return _colab_userdata.get(name)
    except Exception:
        return None

class userdata:  # drop-in for google.colab.userdata
    get = staticmethod(get_api_key)


In [ ]:
# CELL 2 — Imports, Drive, Constants (REPLACEMENT — replaces original Cell 2)
# 6-emotion schema: PositiveAffect, Contentment, NegativeAffect,
#                   Disappointment, Fear, Disgust
# Dropped: Hope, Interest, Surprise, Guilt, Pride, Sadness, Relief

import os, json, time, datetime, warnings
import numpy as np
import pandas as pd
import krippendorff
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, roc_auc_score, matthews_corrcoef,
    precision_score, recall_score, hamming_loss
)
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────
OUTPUT_DIR     = DATA_DIR
HOLDOUT_DIR    = f"{OUTPUT_DIR}/holdout_llm_labels_v4"
TRAIN_DIR      = f"{OUTPUT_DIR}/train_llm_labels_v4"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints_v4"
for d in [HOLDOUT_DIR, TRAIN_DIR, CHECKPOINT_DIR,
          f"{HOLDOUT_DIR}/Run1", f"{HOLDOUT_DIR}/Run2", f"{HOLDOUT_DIR}/Run3",
          f"{TRAIN_DIR}/Run1",   f"{TRAIN_DIR}/Run2",   f"{TRAIN_DIR}/Run3"]:
    os.makedirs(d, exist_ok=True)

# ── 6 merged emotion labels ────────────────────────────────────────────────
EMOTIONS = [
    "PositiveAffect",   # Joy + Love + Gratitude
    "Contentment",      # Contentment only
    "NegativeAffect",   # Anger + Frustration
    "Disappointment",   # Disappointment only
    "Fear",             # Fear only
    "Disgust",          # Disgust only
]

# Which original 16 labels map into each merged label (for remapping old data)
EMOTION_MAP = {
    "PositiveAffect": ["Joy", "Love", "Gratitude"],
    "Contentment":    ["Contentment"],
    "NegativeAffect": ["Anger", "Frustration"],
    "Disappointment": ["Disappointment"],
    "Fear":           ["Fear"],
    "Disgust":        ["Disgust"],
}
DROPPED_EMOTIONS = [
    "Hope", "Interest", "Surprise", "Guilt", "Pride", "Sadness", "Relief"
]

# ── Eval dims (unchanged) ─────────────────────────────────────────────────
EVAL_DIMS = {
    "Trust":          ["Low", "Medium", "High"],
    "Commitment":     ["Low", "Medium", "High"],
    "Recommendation": ["Would Not Recommend", "Neutral", "Would Recommend"],
    "Sentiment":      ["Negative", "Neutral", "Positive"],
}
CLASSES = EMOTIONS + list(EVAL_DIMS.keys())  # 10 total

CATEGORIES = [
    "Baby_Products", "Pet_Supplies", "Beauty_and_Personal_Care",
    "Grocery_and_Gourmet_Food", "Toys_and_Games", "Health_and_Household",
    "Musical_Instruments", "Arts_Crafts_and_Sewing",
    "Home_and_Kitchen", "Sports_and_Outdoors",
]
RUNS = [1, 2, 3]

print("✓ Cell 2 ready")
print(f"  Emotions  : {EMOTIONS}")
print(f"  Eval dims : {list(EVAL_DIMS.keys())}")
print(f"  Classes   : {len(CLASSES)} total")
print(f"  Dropped   : {DROPPED_EMOTIONS}")

✓ Cell 2 ready
  Emotions  : ['PositiveAffect', 'Contentment', 'NegativeAffect', 'Disappointment', 'Fear', 'Disgust']
  Eval dims : ['Trust', 'Commitment', 'Recommendation', 'Sentiment']
  Classes   : 10 total
  Dropped   : ['Hope', 'Interest', 'Surprise', 'Guilt', 'Pride', 'Sadness', 'Relief']


In [ ]:
# CELL 3 — Emotion mapping helper
# Used to remap any existing 16-label data into the new 6-label schema.
# Also used in the prelim labeler to collapse raw GPT output.

def remap_emotions(row, src_cols=None):
    """
    Takes a row (or dict) with original 16 emotion columns (0/1)
    and returns a dict with the 6 merged emotion labels.
    A merged label = 1 if ANY of its source emotions = 1.
    """
    if src_cols is None:
        src_cols = [
            "Joy","Love","Surprise","Pride","Gratitude","Contentment",
            "Relief","Hope","Interest","Anger","Frustration","Disappointment",
            "Sadness","Fear","Guilt","Disgust"
        ]
    out = {}
    for merged_label, sources in EMOTION_MAP.items():
        out[merged_label] = 1 if any(
            int(row.get(s, 0) or 0) == 1 for s in sources
        ) else 0
    return out

def remap_prelim_list(emotion_list):
    """
    Takes a list of original emotion strings e.g. ["Joy", "Frustration"]
    and returns a list of merged label strings e.g. ["PositiveAffect", "NegativeAffect"]
    """
    out = set()
    for merged_label, sources in EMOTION_MAP.items():
        if any(s in emotion_list for s in sources):
            out.add(merged_label)
    return list(out)

# Quick test
test = remap_emotions({"Joy":1,"Frustration":1,"Contentment":0,"Fear":0,"Disgust":0,"Disappointment":0,"Love":0,"Gratitude":0,"Anger":0})
print("Remap test:", test)
print("Expected  : PositiveAffect=1, NegativeAffect=1, rest=0")
print()
test2 = remap_prelim_list(["Joy","Anger","Guilt","Pride"])
print("Prelim remap test:", test2)
print("Expected  : [PositiveAffect, NegativeAffect] (Guilt+Pride dropped)")

Remap test: {'PositiveAffect': 1, 'Contentment': 0, 'NegativeAffect': 1, 'Disappointment': 0, 'Fear': 0, 'Disgust': 0}
Expected  : PositiveAffect=1, NegativeAffect=1, rest=0

Prelim remap test: ['NegativeAffect', 'PositiveAffect']
Expected  : [PositiveAffect, NegativeAffect] (Guilt+Pride dropped)


In [ ]:
# CELL 4 — System prompts (main labeler + prelim screener)
# Key changes from original:
#   1. Only 6 merged emotion labels
#   2. 'Example cues (not exhaustive)' — GPT does not treat as hard checklist
#   3. Cleaner definitions with no disambiguation needed between dropped emotions

SYSTEM_PROMPT_V2 = """Label this Amazon product review.

TASK A — EMOTIONS (multi-label, 0/1). Judge overall tone, not just exact words.

PositiveAffect  — joy, love, delight, loyalty, or gratitude toward product/brand
                  e.g. (not exhaustive): amazing, love it, so happy, my go-to, grateful
Contentment     — calm satisfaction, product simply works, no strong emotion
                  e.g. (not exhaustive): works fine, no complaints, does what it says
NegativeAffect  — anger, frustration, blame, or feeling blocked by product
                  e.g. (not exhaustive): furious, so annoying, keeps failing, scam
Disappointment  — unmet expectations, product fell short of what was hoped for
                  e.g. (not exhaustive): let down, expected more, not worth it
Fear            — safety concern, danger, or health risk from product
                  e.g. (not exhaustive): unsafe, dangerous, health risk, could hurt
Disgust         — physical revulsion toward product
                  e.g. (not exhaustive): gross, disgusting, foul smell, made me gag

TASK B — EVALUATIONS (single label, middle if unclear):
Trust: Low / Medium / High
Commitment: Low / Medium / High
Recommendation: Would Not Recommend / Neutral / Would Recommend
Sentiment: Negative / Neutral / Positive

Return ONLY this JSON:
{
  "emotions": {
    "PositiveAffect": 0, "Contentment": 0, "NegativeAffect": 0,
    "Disappointment": 0, "Fear": 0, "Disgust": 0
  },
  "Trust": "Medium", "Commitment": "Medium",
  "Recommendation": "Neutral", "Sentiment": "Neutral"
}"""

# Prelim screener — short, cheap, just finds candidate reviews
PRELIM_PROMPT_V2 = """Screen this product review for emotions.
Return a JSON array of emotions clearly present.
Emotions: PositiveAffect, Contentment, NegativeAffect, Disappointment, Fear, Disgust
Definitions:
  PositiveAffect = joy, love, delight, gratitude, loyalty
  Contentment    = calm satisfaction, works fine, no complaints
  NegativeAffect = anger, frustration, blame, annoyance
  Disappointment = unmet expectations, let down, fell short
  Fear           = safety concern, danger, health risk
  Disgust        = physical revulsion, gross, foul
Judge overall tone, not just exact words.
Return ONLY a JSON array e.g. ["NegativeAffect","Disappointment"] or []"""

print(f"✓ System prompt v2 ready ({len(SYSTEM_PROMPT_V2)} chars)")
print(f"  Prelim prompt: {len(PRELIM_PROMPT_V2)} chars")

✓ System prompt v2 ready (1535 chars)
  Prelim prompt: 645 chars


In [ ]:
# CELL 5 — Collect targeted new reviews
# Low-star (1-2): NegativeAffect, Disappointment, Fear, Disgust
# High-star (4-5): PositiveAffect, Contentment
# Runtime: ~15-25 min

df_holdout_v3    = pd.read_csv(f"{OUTPUT_DIR}/holdout_raw_v3.csv")
df_train_test_v3 = pd.read_csv(f"{OUTPUT_DIR}/train_test_raw_v3.csv")
all_used_keys = (
    set(df_holdout_v3["review_text"].str[:120]) |
    set(df_train_test_v3["review_text"].str[:120])
)
print(f"Existing — holdout: {len(df_holdout_v3)}, train/test: {len(df_train_test_v3)}")
print(f"Dedup keys to exclude: {len(all_used_keys)}")

TARGETED_SEED_LOW  = 7777
TARGETED_SEED_HIGH = 8888
TARGET_PER_CAT     = 3000

def collect_targeted(seed, star_ratings, label):
    all_records = []
    for cat in CATEGORIES:
        print(f"  [{label}] {cat}...")
        ds = load_dataset(
            "McAuley-Lab/Amazon-Reviews-2023",
            f"raw_review_{cat}",
            split="full", streaming=True, trust_remote_code=True,
        )
        ds_shuf = ds.shuffle(seed=seed, buffer_size=50_000)
        records, seen = [], set()
        for review in ds_shuf:
            if len(records) >= TARGET_PER_CAT: break
            text   = (review.get("text") or "").strip()
            rating = review.get("rating")
            if not text or len(text.split()) < 15: continue
            if rating not in star_ratings: continue
            key = text[:120]
            if key in all_used_keys or key in seen: continue
            seen.add(key)
            records.append({
                "category": cat, "rating": rating,
                "review_title": (review.get("title") or "").strip(),
                "review_text": text,
                "verified_purchase": review.get("verified_purchase"),
                "helpful_votes": review.get("helpful_vote", 0),
                "user_id": review.get("user_id", ""),
                "asin": review.get("asin", ""),
                "timestamp": review.get("timestamp"),
            })
        all_records.extend(records)
        print(f"    -> {len(records)} collected")
    return pd.DataFrame(all_records)

print("\nCollecting 1-2 star reviews...")
df_low_star = collect_targeted(TARGETED_SEED_LOW, [1.0, 2.0], "low-star")
print(f"Low-star total: {len(df_low_star)}")

print("\nCollecting 4-5 star reviews...")
df_high_star = collect_targeted(TARGETED_SEED_HIGH, [4.0, 5.0], "high-star")
print(f"High-star total: {len(df_high_star)}")

df_new_pool = pd.concat([df_low_star, df_high_star]).reset_index(drop=True)
df_new_pool.to_csv(f"{OUTPUT_DIR}/new_pool_v4.csv", index=False)
print(f"\n✓ Pool saved: {len(df_new_pool)} reviews")

Existing — holdout: 1441, train/test: 20411
Dedup keys to exclude: 21849

  [low-star] Baby_Products...
    -> 3000 collected
  [low-star] Pet_Supplies...
    -> 3000 collected
  [low-star] Beauty_and_Personal_Care...
    -> 3000 collected
  [low-star] Grocery_and_Gourmet_Food...
    -> 3000 collected
  [low-star] Toys_and_Games...
    -> 3000 collected
  [low-star] Health_and_Household...
    -> 3000 collected
  [low-star] Musical_Instruments...
    -> 3000 collected
  [low-star] Arts_Crafts_and_Sewing...
    -> 3000 collected
  [low-star] Home_and_Kitchen...
    -> 3000 collected
  [low-star] Sports_and_Outdoors...
    -> 3000 collected
Low-star total: 30000

  [high-star] Baby_Products...
    -> 3000 collected
  [high-star] Pet_Supplies...
    -> 3000 collected
  [high-star] Beauty_and_Personal_Care...
    -> 3000 collected
  [high-star] Grocery_and_Gourmet_Food...
    -> 3000 collected
  [high-star] Toys_and_Games...
    -> 3000 collected
  [high-star] Health_and_Household...
    -

In [ ]:
# CELL 6 — Prelim-label pool (~$0.08, ~8 min with 6 labels)

def get_api_key(name):
    try:
        return userdata.get(name)
    except:
        return os.environ.get(name)

def parse_prelim(text):
    try:
        text = text.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"): text = text[4:]
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return [e for e in parsed if e in EMOTIONS]
    except:
        pass
    return []

client_prelim = OpenAI(api_key=get_api_key("OPENAI_API_KEY"))

def prelim_label_one(review_text, retries=3):
    for attempt in range(retries):
        try:
            resp = client_prelim.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[
                    {"role": "system", "content": PRELIM_PROMPT_V2},
                    {"role": "user",   "content": review_text[:1500]},
                ],
                temperature=0, max_tokens=60,
            )
            return parse_prelim(resp.choices[0].message.content or "")
        except Exception as e:
            if attempt < retries - 1: time.sleep([3, 8][attempt])
    return []

n_low  = min(3500, len(df_low_star))
n_high = min(2500, len(df_high_star))
df_to_prelim = pd.concat([
    df_low_star.sample(n=n_low,  random_state=42),
    df_high_star.sample(n=n_high, random_state=42),
]).reset_index(drop=True)
print(f"Prelim-labeling {len(df_to_prelim)} reviews...")
print(f"Estimated cost: ~${len(df_to_prelim) * 0.000014:.2f}")

PRELIM_CKPT = f"{OUTPUT_DIR}/prelim_labels_v4.csv"
if os.path.exists(PRELIM_CKPT):
    df_done = pd.read_csv(PRELIM_CKPT)
    df_done["prelim_emotions"] = df_done["prelim_emotions"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x)
    done_keys = set(df_done["review_text"].str[:120])
    df_todo   = df_to_prelim[~df_to_prelim["review_text"].str[:120].isin(done_keys)]
    results   = df_done.to_dict("records")
    print(f"  Resuming: {len(df_done)} done, {len(df_todo)} remaining")
else:
    df_todo, results = df_to_prelim, []
    print("  Starting fresh")

def prelim_worker(row):
    return {**row.to_dict(), "prelim_emotions": prelim_label_one(row["review_text"])}

# unpack iterrows first so row is a Series, not a tuple
rows_todo = [row for _, row in df_todo.iterrows()]

with ThreadPoolExecutor(max_workers=15) as ex:
    futures = {ex.submit(prelim_worker, row): i for i, row in enumerate(rows_todo)}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Prelim"):
        results.append(future.result())
        if len(results) % 500 == 0:
            df_ck = pd.DataFrame(results)
            df_ck["prelim_emotions"] = df_ck["prelim_emotions"].apply(json.dumps)
            df_ck.to_csv(PRELIM_CKPT, index=False)

df_prelim_v4 = pd.DataFrame(results)
df_save = df_prelim_v4.copy()
df_save["prelim_emotions"] = df_save["prelim_emotions"].apply(json.dumps)
df_save.to_csv(PRELIM_CKPT, index=False)
print(f"\n✓ Prelim done: {len(df_prelim_v4)} reviews")
print("\nEmotion distribution in pool:")
for em in EMOTIONS:
    count = sum(1 for x in df_prelim_v4["prelim_emotions"] if em in x)
    print(f"  {em:<20}: {count:4d} ({count/len(df_prelim_v4)*100:.1f}%)")

Prelim-labeling 6000 reviews...
Estimated cost: ~$0.08
  Starting fresh


Prelim: 100%|██████████| 6000/6000 [04:07<00:00, 24.22it/s]



✓ Prelim done: 6000 reviews

Emotion distribution in pool:
  PositiveAffect      : 2240 (37.3%)
  Contentment         : 2468 (41.1%)
  NegativeAffect      : 2391 (39.9%)
  Disappointment      : 3796 (63.3%)
  Fear                :  394 (6.6%)
  Disgust             :  273 (4.5%)


In [ ]:
# CELL 7 — Build holdout_v4_raw.csv + train_v4 + test_v4
# holdout_v4_raw.csv = ONE clean file (all new rows, text only, no labels)
#   -> distributed to team for manual labeling
# train_final_v4.csv + test_final_v4.csv = updated train/test sets

all_used_keys_v4 = (
    set(df_holdout_v3["review_text"].str[:120]) |
    set(df_train_test_v3["review_text"].str[:120])
)
df_pool_v4 = df_prelim_v4[
    ~df_prelim_v4["review_text"].str[:120].isin(all_used_keys_v4)
].copy()
df_pool_v4["num_emotions"] = df_pool_v4["prelim_emotions"].apply(len)
print(f"Candidate pool: {len(df_pool_v4)} reviews")

# Target counts per merged label in holdout (from Step 7 human labels, remapped)
# PositiveAffect = Joy(397)+Love(242)+Gratitude(81) overlapping -> est ~580 unique
# Contentment = 746, NegativeAffect = Anger(59)+Frustration(270) -> ~300
# Disappointment = 402, Fear = 78, Disgust = 69
HOLDOUT_COUNTS_V4 = {
    "PositiveAffect": 580,
    "Contentment":    746,
    "NegativeAffect": 300,
    "Disappointment": 402,
    "Fear":            78,
    "Disgust":         69,
}
MIN_HOLDOUT = 120
gaps = {em: MIN_HOLDOUT - cnt for em, cnt in HOLDOUT_COUNTS_V4.items()
        if MIN_HOLDOUT - cnt > 0}

print("\nHoldout top-up needed:")
for em, gap in sorted(gaps.items(), key=lambda x: HOLDOUT_COUNTS_V4[x[0]]):
    print(f"  {em:<20}: currently ~{HOLDOUT_COUNTS_V4[em]} -> target {MIN_HOLDOUT} (need +{gap})")

emotion_counts_h   = dict(HOLDOUT_COUNTS_V4)
holdout_selected   = set()
emotions_by_rarity = sorted(gaps.keys(), key=lambda e: HOLDOUT_COUNTS_V4[e])

print("\nSelecting holdout top-up reviews:")
for rare_em in emotions_by_rarity:
    before = emotion_counts_h[rare_em]
    candidates = df_pool_v4[
        (df_pool_v4["prelim_emotions"].apply(lambda x: rare_em in x)) &
        (~df_pool_v4.index.isin(holdout_selected))
    ].sort_values("num_emotions", ascending=False)
    for idx, row in candidates.iterrows():
        if emotion_counts_h[rare_em] >= MIN_HOLDOUT: break
        holdout_selected.add(idx)
        for em in row["prelim_emotions"]:
            if em in emotion_counts_h: emotion_counts_h[em] += 1
    print(f"  {rare_em:<20}: ~{before} -> ~{emotion_counts_h[rare_em]}")

# Build holdout_v4: old rows + new top-up rows
df_holdout_topup = df_pool_v4.loc[list(holdout_selected)].drop(
    columns=["prelim_emotions", "num_emotions"], errors="ignore"
).reset_index(drop=True)
df_holdout_v4 = pd.concat([df_holdout_v3, df_holdout_topup]).reset_index(drop=True)

# Build train/test v4: add rare-emotion reviews to train pool
used_for_holdout = set(df_holdout_topup["review_text"].str[:120])
df_pool_train = df_pool_v4[
    ~df_pool_v4["review_text"].str[:120].isin(used_for_holdout)
].copy()
train_selected = set()
print("\nTrain top-up:")
for rare_em in EMOTIONS:
    candidates = df_pool_train[
        (df_pool_train["prelim_emotions"].apply(lambda x: rare_em in x)) &
        (~df_pool_train.index.isin(train_selected))
    ].sort_values("num_emotions", ascending=False).head(200)
    train_selected.update(candidates.index.tolist())
    print(f"  {rare_em:<20}: +{len(candidates)}")

df_train_topup = df_pool_train.loc[list(train_selected)].drop(
    columns=["prelim_emotions", "num_emotions"], errors="ignore"
).reset_index(drop=True)
df_train_test_v4 = pd.concat([df_train_test_v3, df_train_topup]).reset_index(drop=True)
train_v4, test_v4 = train_test_split(
    df_train_test_v4, test_size=0.2, random_state=42,
    stratify=df_train_test_v4["category"]
)
train_v4 = train_v4.reset_index(drop=True)
test_v4  = test_v4.reset_index(drop=True)

# Quality checks
h_keys = set(df_holdout_v4["review_text"].str[:120])
t_keys = set(df_train_test_v4["review_text"].str[:120])
print(f"\nQuality checks:")
print(f"  Overlap holdout/train : {len(h_keys & t_keys)} (must be 0)")
print(f"  Duplicates in train   : {df_train_test_v4.duplicated(subset=['review_text']).sum()} (must be 0)")

# Save — holdout is TEXT ONLY (no labels), team will label manually
df_holdout_v4.to_csv(f"{OUTPUT_DIR}/holdout_v4_raw.csv",    index=False)
train_v4.to_csv(f"{OUTPUT_DIR}/train_final_v4.csv",         index=False)
test_v4.to_csv(f"{OUTPUT_DIR}/test_final_v4.csv",           index=False)
df_train_test_v4.to_csv(f"{OUTPUT_DIR}/train_test_raw_v4.csv", index=False)

print(f"\n=== FILES SAVED ===")
print(f"  holdout_v4_raw.csv    : {len(df_holdout_v4)} reviews (text only, no labels)")
print(f"  train_final_v4.csv   : {len(train_v4)} reviews")
print(f"  test_final_v4.csv    : {len(test_v4)} reviews")
print(f"  train_test_raw_v4.csv: {len(df_train_test_v4)} reviews")

Candidate pool: 6000 reviews

Holdout top-up needed:
  Disgust             : currently ~69 -> target 120 (need +51)
  Fear                : currently ~78 -> target 120 (need +42)

Selecting holdout top-up reviews:
  Disgust             : ~69 -> ~120
  Fear                : ~108 -> ~120

Train top-up:
  PositiveAffect      : +200
  Contentment         : +200
  NegativeAffect      : +200
  Disappointment      : +200
  Fear                : +129
  Disgust             : +92

Quality checks:
  Overlap holdout/train : 0 (must be 0)
  Duplicates in train   : 0 (must be 0)

=== FILES SAVED ===
  holdout_v4_raw.csv    : 1504 reviews (text only, no labels)
  train_final_v4.csv   : 17145 reviews
  test_final_v4.csv    : 4287 reviews
  train_test_raw_v4.csv: 21432 reviews


In [ ]:
# CELL 8 — Gate check: confirms manual labeling is complete before continuing
# This cell will throw an error if final_holdout_v2.csv does not exist.
# Once your team has finished labeling and uploaded the file, run this cell.
# If it prints the green check, you are good to continue.

gate_path = f"{OUTPUT_DIR}/final_holdout_v2.csv"
assert os.path.exists(gate_path), (
    "\n\n"
    "STOP — final_holdout_v2.csv not found in your Drive.\n"
    "Complete manual labeling of holdout_v4_raw.csv first,\n"
    "then save the labeled version as final_holdout_v2.csv and re-run this cell."
)
df_check = pd.read_csv(gate_path)
missing_cols = [c for c in EMOTIONS + list(EVAL_DIMS.keys()) if c not in df_check.columns]
assert not missing_cols, f"Missing label columns in final_holdout_v2.csv: {missing_cols}"
print(f"✓ final_holdout_v2.csv found: {len(df_check)} labeled reviews")
print(f"\nLabel distribution:")
for em in EMOTIONS:
    c = df_check[em].sum()
    print(f"  {em:<22}: {int(c)} ({c/len(df_check)*100:.1f}%)")
print("\n✓ Gate passed — continue to Cell 9.")

✓ final_holdout_v2.csv found: 1504 labeled reviews

Label distribution:
  PositiveAffect        : 1010 (67.2%)
  Contentment           : 1045 (69.5%)
  NegativeAffect        : 449 (29.9%)
  Disappointment        : 589 (39.2%)
  Fear                  : 132 (8.8%)
  Disgust               : 92 (6.1%)

✓ Gate passed — continue to Cell 9.


In [ ]:
# CELL 9 — Parse + classify helpers (shared client + connection pool)

import httpx

def get_api_key(name):
    try:
        return userdata.get(name)
    except:
        return os.environ.get(name)

# Single shared client with explicit connection pool — reused across all workers
_SHARED_CLIENT = OpenAI(
    api_key=get_api_key("OPENAI_API_KEY"),
    http_client=httpx.Client(
        limits=httpx.Limits(
            max_connections=100,
            max_keepalive_connections=50
        )
    )
)

def parse_response_v2(text):
    try:
        text = text.strip()
        if "</think>" in text: text = text.split("</think>")[-1].strip()
        if "```" in text:
            text = "\n".join(l for l in text.split("\n")
                             if not l.strip().startswith("```")).strip()
        start = text.find("{")
        if start == -1: return None, None, "No JSON"
        depth, end = 0, start
        for i, ch in enumerate(text[start:], start):
            if ch == "{": depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0: end = i + 1; break
        if depth > 0: text += "}" * depth
        data   = json.loads(text[start:end] if depth == 0 else text[start:])
        em_out = {em: int(bool(data.get("emotions", {}).get(em, 0))) for em in EMOTIONS}
        ev_out = {}
        for dim, levels in EVAL_DIMS.items():
            val = data.get(dim, "")
            ev_out[dim] = val if val in levels else levels[len(levels)//2]
        return em_out, ev_out, None
    except Exception as e:
        return None, None, str(e)

def classify_one(row_id, review_text, retries=3):
    for attempt in range(retries):
        try:
            t0   = time.perf_counter()
            resp = _SHARED_CLIENT.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT_V2},
                    {"role": "user",   "content": review_text[:2000]},
                ],
                temperature=0, max_tokens=200,
                response_format={"type": "json_object"},
            )
            raw     = resp.choices[0].message.content or ""
            latency = time.perf_counter() - t0
            em, ev, err = parse_response_v2(raw)
            if err: raise ValueError(err)
            cost = (resp.usage.prompt_tokens/1e6)*0.40 + (resp.usage.completion_tokens/1e6)*1.60
            rec  = {"id": row_id, "review_text": review_text,
                    "error": None, "cost_usd": cost, "latency_s": latency}
            for e in EMOTIONS:  rec[e] = em.get(e, 0)
            for d in EVAL_DIMS: rec[d] = ev.get(d, EVAL_DIMS[d][len(EVAL_DIMS[d])//2])
            return rec
        except Exception as e:
            if attempt < retries - 1: time.sleep([5, 15, 30][attempt])
    rec = {"id": row_id, "review_text": review_text,
           "error": str(e), "cost_usd": 0, "latency_s": 0}
    for e in EMOTIONS:  rec[e] = 0
    for d in EVAL_DIMS: rec[d] = EVAL_DIMS[d][len(EVAL_DIMS[d])//2]
    return rec

print("✓ Shared client + connection pool ready")
print(f"  Max connections: 100 | Keepalive: 50")

✓ Shared client + connection pool ready
  Max connections: 100 | Keepalive: 50


In [ ]:
# CELL 10 — Label holdout_v4 with GPT (3 runs) + majority vote + evaluate
# Ground truth = final_holdout_v2.csv (human labels)
# Target: Macro F1 >= 0.80, ideally >= 0.90
# Cost: ~$1.50 | Time: ~10 min

def run_holdout(run=1, max_workers=20):
    run_dir  = f"{HOLDOUT_DIR}/Run{run}"
    os.makedirs(run_dir, exist_ok=True)
    ckpt     = f"{CHECKPOINT_DIR}/holdout_v4_run{run}.pkl"
    out_path = f"{run_dir}/holdout_v4_run{run}.pkl"
    df_h     = pd.read_csv(f"{OUTPUT_DIR}/holdout_v4_raw.csv")
    df_h["id"] = df_h.index
    results, done_ids = [], set()
    if os.path.exists(ckpt):
        ck = pd.read_pickle(ckpt); ok = ck[ck["error"].isna()]
        results = ok.to_dict("records"); done_ids = set(ok["id"].values)
        print(f"  Resuming run{run}: {len(results)} done")
    df_todo = df_h[~df_h["id"].isin(done_ids)]
    rows = [r for _, r in df_todo.iterrows()]
    lock = __import__("threading").Lock()
    print(f"  Run {run}: {len(df_h)} reviews | {len(df_todo)} remaining")
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(classify_one, r["id"], r["review_text"]): r["id"] for r in rows}
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"run{run}"):
            with lock:
                results.append(future.result())
                if len(results) % 200 == 0:
                    pd.DataFrame(results).to_pickle(ckpt)
                    cost = sum(r["cost_usd"] for r in results if not r.get("error"))
                    print(f"  [{len(results)}/{len(df_h)}] cost: ${cost:.3f}")
    df_out = pd.DataFrame(results)
    df_out.to_pickle(ckpt); df_out.to_pickle(out_path)
    ok = df_out[df_out["error"].isna()]
    print(f"  Run {run} done: {len(ok)}/{len(df_h)} ok | cost: ${ok['cost_usd'].sum():.3f}")
    return df_out

for run in RUNS:
    run_holdout(run=run)

# Majority vote
run_dfs = {}
for run in RUNS:
    path = f"{HOLDOUT_DIR}/Run{run}/holdout_v4_run{run}.pkl"
    df = pd.read_pickle(path)
    run_dfs[run] = df[df["error"].isna()].set_index("id")
common_ids = sorted(set.intersection(*[set(run_dfs[r].index) for r in RUNS]))
print(f"\nMajority vote across {len(RUNS)} runs: {len(common_ids)} reviews")
mv_results = []
for row_id in tqdm(common_ids, desc="Majority vote"):
    rows   = {r: run_dfs[r].loc[row_id] for r in RUNS}
    record = {"id": row_id, "review_text": rows[1]["review_text"]}
    for em in EMOTIONS:
        votes = sum(int(rows[r].get(em, 0) or 0) for r in RUNS)
        record[em] = 1 if votes > len(RUNS)/2 else 0
    for dim, levels in EVAL_DIMS.items():
        lv = {l: 0 for l in levels}
        for r in RUNS:
            v = rows[r].get(dim)
            if v in lv: lv[v] += 1
        record[dim] = max(lv, key=lv.get)
    mv_results.append(record)
df_mv = pd.DataFrame(mv_results)
df_mv.to_csv(f"{HOLDOUT_DIR}/majority_vote_holdout_v4.csv", index=False)
print("✓ Majority vote saved")

# Evaluate vs human labels
df_gt = pd.read_csv(f"{OUTPUT_DIR}/final_holdout_v2.csv")
merged = df_mv.merge(df_gt[["review_text"] + EMOTIONS], on="review_text")
print(f"\nMatched for evaluation: {len(merged)} reviews")
print(f"\n{'='*60}")
print("GPT-4.1-mini v2 vs Human Labels")
print(f"{'='*60}")
print(f"{'Emotion':<22} {'F1':>7} {'Prec':>7} {'Rec':>7} {'Support':>8}")
print("-"*50)
f1s = []
for em in EMOTIONS:
    col_m = em + "_x" if em + "_x" in merged.columns else em
    col_h = em + "_y" if em + "_y" in merged.columns else em
    yt = merged[col_h].values.astype(int)
    yp = merged[col_m].values.astype(int)
    f1 = f1_score(yt, yp, zero_division=0)
    pr = precision_score(yt, yp, zero_division=0)
    rc = recall_score(yt, yp, zero_division=0)
    f1s.append(f1)
    print(f"{em:<22} {f1:>7.4f} {pr:>7.4f} {rc:>7.4f} {int(yt.sum()):>8}")
macro_f1 = float(np.mean(f1s))
print("-"*50)
print(f"{'Macro avg':<22} {macro_f1:>7.4f}")
print(f"\n  Macro F1 : {macro_f1:.4f}")
if macro_f1 >= 0.90:
    print("  ✓ EXCELLENT — above 0.90, proceed to label train/test")
elif macro_f1 >= 0.80:
    print("  ✓ GOOD — above 0.80, proceed to label train/test")
else:
    print("  ✗ Below 0.80 — consider refining system prompt before Cell 11")

  Resuming run1: 1504 done
  Run 1: 1504 reviews | 0 remaining


run1: 0it [00:00, ?it/s]


  Run 1 done: 1504/1504 ok | cost: $0.646
  Resuming run2: 1504 done
  Run 2: 1504 reviews | 0 remaining


run2: 0it [00:00, ?it/s]


  Run 2 done: 1504/1504 ok | cost: $0.646
  Resuming run3: 1504 done
  Run 3: 1504 reviews | 0 remaining


run3: 0it [00:00, ?it/s]


  Run 3 done: 1504/1504 ok | cost: $0.646

Majority vote across 3 runs: 1504 reviews


Majority vote: 100%|██████████| 1504/1504 [00:00<00:00, 4451.28it/s]


✓ Majority vote saved

Matched for evaluation: 1504 reviews

GPT-4.1-mini v2 vs Human Labels
Emotion                     F1    Prec     Rec  Support
--------------------------------------------------
PositiveAffect          0.9921  0.9950  0.9892     1014
Contentment             0.9426  0.9160  0.9707      989
NegativeAffect          0.8877  0.8985  0.8772      464
Disappointment          0.9075  0.9374  0.8794      630
Fear                    0.8014  0.8561  0.7533      150
Disgust                 0.8203  0.9570  0.7177      124
--------------------------------------------------
Macro avg               0.8919

  Macro F1 : 0.8919
  ✓ GOOD — above 0.80, proceed to label train/test


In [ ]:
# CELL 11 — Label train + test v4 with GPT (Step 8 redo)
# Cost: ~$22 | Checkpoints every 500 reviews — safe to interrupt and resume

CHECKPOINT_DIR_TRAIN = f"{OUTPUT_DIR}/checkpoints_train_v4"
os.makedirs(CHECKPOINT_DIR_TRAIN, exist_ok=True)

def run_dataset(dataset_name, csv_path, run=1, max_workers=20):
    base     = f"{dataset_name}_gpt41mini_v4_run{run}"
    ckpt     = f"{CHECKPOINT_DIR_TRAIN}/{base}.pkl"
    run_dir  = f"{TRAIN_DIR}/Run{run}"
    out_path = f"{run_dir}/{base}.pkl"
    os.makedirs(run_dir, exist_ok=True)
    df = pd.read_csv(csv_path); df["id"] = df.index
    results, done_ids = [], set()
    if os.path.exists(ckpt):
        ck = pd.read_pickle(ckpt); ok = ck[ck["error"].isna()]
        results = ok.to_dict("records"); done_ids = set(ok["id"].values)
        print(f"  Resuming {dataset_name} run{run}: {len(results)} done, {len(df)-len(results)} remaining")
    else:
        print(f"  Starting {dataset_name} run{run}: {len(df)} reviews")
    df_todo = df[~df["id"].isin(done_ids)]
    rows = [r for _, r in df_todo.iterrows()]
    lock = __import__("threading").Lock()
    start = datetime.datetime.now()
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(classify_one, r["id"], r["review_text"]): r["id"] for r in rows}
        completed = 0
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"{dataset_name} run{run}"):
            with lock:
                results.append(future.result())
                completed += 1
                if completed % 500 == 0:
                    pd.DataFrame(results).to_pickle(ckpt)
                    cost    = sum(r["cost_usd"] for r in results if not r.get("error"))
                    elapsed = (datetime.datetime.now() - start).total_seconds()
                    print(f"  [{completed}/{len(df_todo)}] cost: ${cost:.2f} | {elapsed/60:.1f}m")
    df_out = pd.DataFrame(results)
    df_out.to_pickle(ckpt); df_out.to_pickle(out_path)
    df_out.to_csv(out_path.replace(".pkl", ".csv"), index=False)
    ok = df_out[df_out["error"].isna()]
    print(f"  ✓ {dataset_name} run{run}: {len(ok)}/{len(df_out)} ok | cost: ${ok['cost_usd'].sum():.2f}")
    return df_out

def majority_vote_dataset(dataset_name):
    run_dfs = {}
    for run in RUNS:
        path = f"{TRAIN_DIR}/Run{run}/{dataset_name}_gpt41mini_v4_run{run}.pkl"
        df = pd.read_pickle(path)
        run_dfs[run] = df[df["error"].isna()].set_index("id")
    common_ids = sorted(set.intersection(*[set(run_dfs[r].index) for r in RUNS]))
    print(f"  {dataset_name}: {len(common_ids)} common across {len(RUNS)} runs")
    results = []
    for row_id in tqdm(common_ids, desc=f"{dataset_name} mv"):
        rows   = {r: run_dfs[r].loc[row_id] for r in RUNS}
        record = {"id": row_id, "review_text": rows[1]["review_text"]}
        for em in EMOTIONS:
            votes = sum(int(rows[r].get(em, 0) or 0) for r in RUNS)
            record[em] = 1 if votes > len(RUNS)/2 else 0
        for dim, levels in EVAL_DIMS.items():
            lv = {l: 0 for l in levels}
            for r in RUNS:
                v = rows[r].get(dim)
                if v in lv: lv[v] += 1
            record[dim] = max(lv, key=lv.get)
        results.append(record)
    df_mv = pd.DataFrame(results)
    out = f"{TRAIN_DIR}/{dataset_name}_v4_final_labeled.pkl"
    df_mv.to_pickle(out)
    df_mv.to_csv(out.replace(".pkl", ".csv"), index=False)
    print(f"  ✓ Saved: {len(df_mv)} labeled reviews")
    print("  Label distribution:")
    for em in EMOTIONS:
        c = df_mv[em].sum()
        print(f"    {em:<20}: {c:5d} ({c/len(df_mv)*100:.1f}%)")
    return df_mv

TRAIN_CSV = f"{OUTPUT_DIR}/train_final_v4.csv"
TEST_CSV  = f"{OUTPUT_DIR}/test_final_v4.csv"
print(f"Train: {len(pd.read_csv(TRAIN_CSV))} | Test: {len(pd.read_csv(TEST_CSV))}")
print()

for run in RUNS:
    run_dataset("train", TRAIN_CSV, run=run, max_workers=20)
df_train_labeled = majority_vote_dataset("train")

for run in RUNS:
    run_dataset("test", TEST_CSV, run=run, max_workers=20)
df_test_labeled = majority_vote_dataset("test")

print("\n✓ Step 8 v4 complete")

Train: 17145 | Test: 4287

  Resuming train run1: 17145 done, 0 remaining


train run1: 0it [00:00, ?it/s]


  ✓ train run1: 17145/17145 ok | cost: $7.36
  Resuming train run2: 1000 done, 16145 remaining


train run2:   3%|▎         | 502/16145 [00:41<22:14, 11.72it/s]

  [500/16145] cost: $0.49 | 0.7m


...

  [4000/4287] cost: $1.32 | 5.1m


test run3: 100%|██████████| 4287/4287 [05:53<00:00, 12.13it/s]


  ✓ test run3: 4287/4287 ok | cost: $1.42
  test: 4287 common across 3 runs


test mv: 100%|██████████| 4287/4287 [00:00<00:00, 4400.33it/s]


  ✓ Saved: 4287 labeled reviews
  Label distribution:
    PositiveAffect      :  2933 (68.4%)
    Contentment         :  2975 (69.4%)
    NegativeAffect      :  1194 (27.9%)
    Disappointment      :  2060 (48.1%)
    Fear                :   162 (3.8%)
    Disgust             :   160 (3.7%)

✓ Step 8 v4 complete


In [ ]:
# CELL 12 — Load + prepare data for RoBERTa

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset as HFDataset
import warnings; warnings.filterwarnings('ignore')

EVAL_BINARIZE = {
    "Trust":          {"High": 1, "Medium": 0, "Low": 0},
    "Commitment":     {"High": 1, "Medium": 0, "Low": 0},
    "Recommendation": {"Would Recommend": 1, "Neutral": 0, "Would Not Recommend": 0},
    "Sentiment":      {"Positive": 1, "Neutral": 0, "Negative": 0},
}

print("Loading v4 labeled data...")
df_train_full = pd.read_csv(f"{TRAIN_DIR}/train_v4_final_labeled.csv")
df_test       = pd.read_csv(f"{TRAIN_DIR}/test_v4_final_labeled.csv")
df_holdout    = pd.read_csv(f"{OUTPUT_DIR}/final_holdout_v2.csv")

for dim, mapping in EVAL_BINARIZE.items():
    for df in [df_train_full, df_test, df_holdout]:
        if dim in df.columns:
            df[dim] = df[dim].map(mapping).fillna(0).astype(int)

print(f"  Train: {len(df_train_full)} | Test: {len(df_test)} | Holdout: {len(df_holdout)}")
missing = [c for c in CLASSES if c not in df_holdout.columns]
if missing: print(f"  WARNING missing cols in holdout: {missing}")

# Train/val split
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
    tidx, vidx = next(msss.split(df_train_full, df_train_full[EMOTIONS].values))
    df_train, df_val = df_train_full.iloc[tidx].copy(), df_train_full.iloc[vidx].copy()
    print("  Stratified split")
except ImportError:
    df_train, df_val = train_test_split(df_train_full, test_size=0.1, random_state=42)
    print("  Random split")
print(f"  Train: {len(df_train)} | Val: {len(df_val)}")

train_labels   = df_train[CLASSES].values.astype(np.float32)
val_labels     = df_val[CLASSES].values.astype(np.float32)
test_labels    = df_test[CLASSES].values.astype(np.float32)
holdout_labels = df_holdout[CLASSES].values.astype(np.float32)

# Pos weights — cap at 10x to avoid instability
pos_counts = train_labels.sum(axis=0)
neg_counts = len(df_train) - pos_counts
pos_weight = np.clip(neg_counts / np.maximum(pos_counts, 1), 1.0, 10.0)
print("\nClass weights:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:<22}: {pos_weight[i]:.2f}  (support: {int(pos_counts[i])})")

# Device
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_fp16 = torch.cuda.is_available()
use_bf16 = False
print(f"\nDevice: {device}")

# Tokenize at MAX_LENGTH=160
MAX_LENGTH = 160
tokenizer  = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(texts, labels):
    enc = tokenizer(texts.tolist(), truncation=True, padding="max_length",
                    max_length=MAX_LENGTH, return_tensors=None)
    return HFDataset.from_dict({"input_ids": enc["input_ids"],
                                "attention_mask": enc["attention_mask"],
                                "labels": labels.tolist()})

print(f"\nTokenizing at MAX_LENGTH={MAX_LENGTH}...")
train_dataset   = tokenize(df_train["review_text"].values,   train_labels)
val_dataset     = tokenize(df_val["review_text"].values,     val_labels)
test_dataset    = tokenize(df_test["review_text"].values,    test_labels)
holdout_dataset = tokenize(df_holdout["review_text"].values, holdout_labels)
print("✓ Tokenization done")

pw_tensor = torch.tensor(pos_weight, dtype=torch.float32)

def predict(dataset, model, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size)
    all_probs = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            ids  = torch.stack(batch["input_ids"],      dim=1).to(device)
            mask = torch.stack(batch["attention_mask"], dim=1).to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            all_probs.append(torch.sigmoid(out.logits).cpu().numpy())
    return np.vstack(all_probs)

def evaluate_and_print(y_true, y_pred, y_prob, config_name):
    print(f"\n{'='*60}")
    print(f"HOLDOUT EVALUATION — {config_name}")
    print(f"{'='*60}")
    print(f"{'Class':<22} {'F1':>7} {'AUC':>7} {'Prec':>7} {'Rec':>7} {'Supp':>6}")
    print("-"*55)
    f1s = []
    for i, cls in enumerate(CLASSES):
        yt, yp, ypr = y_true[:,i], y_pred[:,i], y_prob[:,i]
        f1 = f1_score(yt, yp, zero_division=0)
        pr = precision_score(yt, yp, zero_division=0)
        rc = recall_score(yt, yp, zero_division=0)
        try:    auc = roc_auc_score(yt, ypr)
        except: auc = 0.0
        f1s.append(f1)
        print(f"{cls:<22} {f1:>7.4f} {auc:>7.4f} {pr:>7.4f} {rc:>7.4f} {int(yt.sum()):>6}")
    macro = float(np.mean(f1s))
    print("-"*55)
    print(f"{'Macro avg':<22} {macro:>7.4f}")
    print(f"\n  Macro F1 : {macro:.4f}")
    print(f"  Micro F1 : {f1_score(y_true, y_pred, average='micro', zero_division=0):.4f}")
    return macro

print("\n✓ Data + helpers ready for training")

Loading v4 labeled data...
  Train: 17145 | Test: 4287 | Holdout: 1504
  Stratified split
  Train: 15430 | Val: 1715

Class weights:
  PositiveAffect        : 1.00  (support: 10396)
  Contentment           : 1.00  (support: 10600)
  NegativeAffect        : 2.49  (support: 4420)
  Disappointment        : 1.07  (support: 7456)
  Fear                  : 10.00  (support: 607)
  Disgust               : 10.00  (support: 526)
  Trust                 : 1.00  (support: 8187)
  Commitment            : 2.17  (support: 4861)
  Recommendation        : 1.00  (support: 8263)
  Sentiment             : 1.00  (support: 8501)

Device: cuda



Tokenizing at MAX_LENGTH=160...
✓ Tokenization done

✓ Data + helpers ready for training


In [ ]:
# CELL 13 — Custom trainers + run_config

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out    = model(**inputs)
        loss   = nn.BCEWithLogitsLoss(
            pos_weight=pw_tensor.to(out.logits.device)
        )(out.logits, labels.float())
        return (loss, out) if return_outputs else loss

class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out    = model(**inputs)
        probs  = torch.sigmoid(out.logits)
        bce    = nn.functional.binary_cross_entropy_with_logits(
            out.logits, labels.float(), reduction="none",
            pos_weight=pw_tensor.to(out.logits.device)
        )
        p_t  = probs * labels + (1 - probs) * (1 - labels)
        loss = ((1 - p_t) ** 2.0 * bce).mean()
        return (loss, out) if return_outputs else loss

all_results = {}

def run_config(config_name, model_name, learning_rate, batch_size,
               gradient_accumulation, num_epochs, threshold=0.3,
               trainer_class=WeightedTrainer):
    print(f"\n{'='*55}")
    print(f"CONFIG: {config_name}")
    print(f"  Model    : {model_name}")
    print(f"  LR       : {learning_rate}")
    print(f"  Batch    : {batch_size} x {gradient_accumulation} = {batch_size*gradient_accumulation}")
    print(f"  Epochs   : {num_epochs} | MaxLen: {MAX_LENGTH} | Threshold: {threshold}")
    print(f"  Trainer  : {trainer_class.__name__}")
    print(f"{'='*55}")
    save_path = f"{OUTPUT_DIR}/roberta_models_v4/{config_name}"
    os.makedirs(save_path, exist_ok=True)
    model = RobertaForSequenceClassification.from_pretrained(
        model_name, num_labels=len(CLASSES),
        problem_type="multi_label_classification"
    ).to(device)
    args = TrainingArguments(
        output_dir=save_path, num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation,
        learning_rate=learning_rate, weight_decay=0.01,
        warmup_ratio=0.1, optim="adamw_torch",
        lr_scheduler_type="linear", max_grad_norm=1.0,
        eval_strategy="no", save_strategy="no",
        logging_steps=50, logging_first_step=True,
        report_to="none", fp16=use_fp16, bf16=use_bf16,
        seed=42, data_seed=42, dataloader_num_workers=2,
        dataloader_pin_memory=True, remove_unused_columns=False,
    )
    trainer = trainer_class(model=model, args=args,
                            train_dataset=train_dataset, callbacks=[])
    result = trainer.train()
    print(f"  Runtime: {result.metrics['train_runtime']:.0f}s | Loss: {result.metrics['train_loss']:.4f}")
    trainer.save_model(save_path + "/best_model")
    tokenizer.save_pretrained(save_path + "/best_model")
    probs    = predict(holdout_dataset, model)
    preds    = (probs >= threshold).astype(int)
    macro_f1 = evaluate_and_print(holdout_labels, preds, probs, config_name)
    all_results[config_name] = macro_f1
    del model
    import gc; gc.collect(); torch.cuda.empty_cache()
    return macro_f1

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("✓ Trainers and run_config ready")

✓ Trainers and run_config ready


In [ ]:
# CELL 14 — Run all configs
# Config 1: baseline — 2e-5 LR, weighted BCE, MAX_LENGTH=160
# Config 2: higher LR — 5e-5 (best in original Step 9), weighted BCE
# Config 3: focal loss — 5e-5 LR, focal loss
# Config 4: roberta-base — faster, smaller, sanity check

run_config(
    config_name="config1_baseline",
    model_name="roberta-large",
    learning_rate=2e-5, batch_size=8, gradient_accumulation=8,
    num_epochs=4, threshold=0.3, trainer_class=WeightedTrainer,
)

run_config(
    config_name="config2_higher_lr",
    model_name="roberta-large",
    learning_rate=5e-5, batch_size=8, gradient_accumulation=8,
    num_epochs=4, threshold=0.3, trainer_class=WeightedTrainer,
)

run_config(
    config_name="config3_focal_loss",
    model_name="roberta-large",
    learning_rate=5e-5, batch_size=8, gradient_accumulation=8,
    num_epochs=4, threshold=0.3, trainer_class=FocalLossTrainer,
)

run_config(
    config_name="config4_roberta_base",
    model_name="roberta-base",
    learning_rate=5e-5, batch_size=32, gradient_accumulation=2,
    num_epochs=4, threshold=0.3, trainer_class=WeightedTrainer,
)

print(f"\n{'='*55}")
print("ALL CONFIGS COMPLETE")
print(f"{'='*55}")
for cfg, f1 in sorted(all_results.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(f1 * 40)
    print(f"  {cfg:<30} {f1:.4f}  {bar}")
best = max(all_results, key=all_results.get)
print(f"\n  Best: {best} — Macro F1 = {all_results[best]:.4f}")
if all_results[best] >= 0.80:
    print("  ✓ TARGET MET: above 0.80")
else:
    print("  Below 0.80 — see Cell 15 for comparison and write-up framing")


CONFIG: config1_baseline
  Model    : roberta-large
  LR       : 2e-05
  Batch    : 8 x 8 = 64
  Epochs   : 4 | MaxLen: 160 | Threshold: 0.3
  Trainer  : WeightedTrainer


RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
1,6.659758
50,6.250361
100,4.114608
150,2.828873
200,2.251145
250,2.076302
300,1.795233
350,1.723450
400,1.745480
450,1.704068


  Runtime: 1666s | Loss: 1.8750



HOLDOUT EVALUATION — config1_baseline
Class                       F1     AUC    Prec     Rec   Supp
-------------------------------------------------------
PositiveAffect          0.9453  0.9794  0.9128  0.9803   1014
Contentment             0.9448  0.9831  0.9062  0.9869    989
NegativeAffect          0.8689  0.9595  0.8527  0.8858    464
Disappointment          0.8947  0.9722  0.8674  0.9238    630
Fear                    0.7368  0.9462  0.7273  0.7467    150
Disgust                 0.7295  0.9655  0.7417  0.7177    124
Trust                   0.8945  0.9275  0.8256  0.9758    786
Commitment              0.8842  0.9532  0.8911  0.8773    709
Recommendation          0.9300  0.9490  0.9270  0.9330    926
Sentiment               0.9526  0.9846  0.9249  0.9820    890
-------------------------------------------------------
Macro avg               0.8781

  Macro F1 : 0.8781
  Micro F1 : 0.9131

CONFIG: config2_higher_lr
  Model    : roberta-large
  LR       : 5e-05
  Batch    : 8 x 8 = 6

...


HOLDOUT EVALUATION — config3_focal_loss
Class                       F1     AUC    Prec     Rec   Supp
-------------------------------------------------------
PositiveAffect          0.9368  0.9853  0.8858  0.9941   1014
Contentment             0.9337  0.9769  0.8836  0.9899    989
NegativeAffect          0.8531  0.9567  0.8101  0.9009    464
Disappointment          0.8811  0.9712  0.8197  0.9524    630
Fear                    0.7226  0.9341  0.7000  0.7467    150
Disgust                 0.7179  0.9718  0.6577  0.7903    124
Trust                   0.8795  0.9210  0.7947  0.9847    786
Commitment              0.8795  0.9572  0.8500  0.9111    709
Recommendation          0.9287  0.9523  0.9151  0.9428    926
Sentiment               0.9460  0.9858  0.9106  0.9843    890
-------------------------------------------------------
Macro avg               0.8679

  Macro F1 : 0.8679
  Micro F1 : 0.9033

CONFIG: config4_roberta_base
  Model    : roberta-base
  LR       : 5e-05
  Batch    : 32 x 

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
1,1.622231
50,1.525506
100,1.000401
150,0.828043
200,0.711739
250,0.623249
300,0.561147
350,0.547881
400,0.521464
450,0.527189


  Runtime: 458s | Loss: 0.5422



HOLDOUT EVALUATION — config4_roberta_base
Class                       F1     AUC    Prec     Rec   Supp
-------------------------------------------------------
PositiveAffect          0.9380  0.9768  0.8959  0.9842   1014
Contentment             0.9445  0.9832  0.9039  0.9889    989
NegativeAffect          0.8405  0.9529  0.7996  0.8858    464
Disappointment          0.8742  0.9636  0.8321  0.9206    630
Fear                    0.7040  0.9278  0.6608  0.7533    150
Disgust                 0.7323  0.9714  0.7154  0.7500    124
Trust                   0.8857  0.9177  0.8143  0.9707    786
Commitment              0.8741  0.9498  0.8669  0.8815    709
Recommendation          0.9197  0.9472  0.9119  0.9276    926
Sentiment               0.9371  0.9817  0.9057  0.9708    890
-------------------------------------------------------
Macro avg               0.8650

  Macro F1 : 0.8650
  Micro F1 : 0.9013

ALL CONFIGS COMPLETE
  config1_baseline               0.8781  ████████████████████████████

In [ ]:
# CELL 15 — Final comparison table (original 16-label vs new 6-label system)

print(f"\n{'='*70}")
print("FINAL COMPARISON — 16-label system vs 6-label system")
print(f"{'='*70}")
print(f"{'System':<35} {'Macro F1':>10} {'Labels':>8} {'Notes'}")
print("-"*70)

original = [
    ("GPT-4.1-mini v1 vs human",    0.6067, 16, "original prompt and 16 label"),
    ("RoBERTa config1 (baseline)",  0.5690, 16, "original Step 9"),
    ("RoBERTa config2 (5e-5 LR)",   0.6000, 16, "best original Step 9"),
    ("RoBERTa config4 (256 tokens)",0.5977, 16, "original Step 9"),
]
for name, f1, n_labels, note in original:
    print(f"  {name:<33} {f1:>10.4f} {n_labels:>8}   {note}")

print()
for cfg, f1 in sorted(all_results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cfg:<33} {f1:>10.4f} {'6':>8}   new 6-label system, weighted loss, ML=160")

best_orig = 0.6000
best_new  = max(all_results.values()) if all_results else 0
print("-"*70)
print(f"  Improvement: {best_orig:.4f} -> {best_new:.4f} ({best_new-best_orig:+.4f})")
print()
print("CHANGES MADE:")
print("  1. Consolidated 16 emotions -> 6 merged labels (drop undetectable,")
print("     merge confused pairs — reduces label noise and macro F1 dilution)")
print("  2. System prompt: example cues not exhaustive checklist")
print("  3. MAX_LENGTH: 128 -> 160 tokens (captures more review context)")
print("  4. Targeted data augmentation for underrepresented emotion classes")


FINAL COMPARISON — 16-label system vs 6-label system
System                                Macro F1   Labels Notes
----------------------------------------------------------------------
  GPT-4.1-mini v1 vs human              0.6067       16   original prompt and 16 label
  RoBERTa config1 (baseline)            0.5690       16   original Step 9
  RoBERTa config2 (5e-5 LR)             0.6000       16   best original Step 9
  RoBERTa config4 (256 tokens)          0.5977       16   original Step 9

  config1_baseline                      0.8781        6   new 6-label system, weighted loss, ML=160
  config2_higher_lr                     0.8771        6   new 6-label system, weighted loss, ML=160
  config3_focal_loss                    0.8679        6   new 6-label system, weighted loss, ML=160
  config4_roberta_base                  0.8650        6   new 6-label system, weighted loss, ML=160
----------------------------------------------------------------------
  Improvement: 0.6000 -> 0.